# Module 5 · Capstone, Decision Guide & Wrap-up
**⏱ 15 minutes in session** · 2:45 – 3:00 (the full capstone is a take-home exercise)

## 5.1 Which tool, when?

| Your situation | Start with | Library |
|---|---|---|
| Need to state & check assumptions, get an ATE, run robustness checks | Graph → identify → estimate → **refute** | **DoWhy** |
| Observational data, want **who benefits** + confidence intervals | `LinearDML` / `CausalForestDML` / `DRLearner` | **EconML** |
| **Randomized** campaign / A/B test, want to **target** | Meta-learners, uplift forests, Qini/AUUC | **CausalML** (or EconML) |
| Suspected **unmeasured confounding** + a valid instrument | IV estimators (`OrthoIV`, `DRIV`), DoWhy IV | **EconML / DoWhy** |
| **Don't know the graph** | PC / GES / LiNGAM / FCI + domain knowledge | **causal-learn** |
| Root-cause / "why did this metric change?" / counterfactuals on an SCM | Graphical causal models | **DoWhy `gcm`** |

## 5.2 An end-to-end causal workflow (and how it maps to model governance)

| Step | Activity | Governance artifact (model-risk documentation) |
|---|---|---|
| 1. **Frame** | Define the decision, treatment, outcome, population, time horizon, estimand (ATE/ATT/CATE) | Business-use statement |
| 2. **Assume** | Draw the DAG with domain experts; use discovery as a *hypothesis generator* | Assumptions log, reviewed DAG |
| 3. **Identify** | Backdoor / IV / frontdoor via DoWhy; check positivity/overlap | Identification memo |
| 4. **Estimate** | ≥2 estimator families; DML/DR for robustness; CATE where decisions differ by segment | Methodology section |
| 5. **Refute** | Placebo, random cause, subsets, **sensitivity to hidden confounding** | Robustness & limitations section |
| 6. **Decide** | CATE vs cost → policy; interpretable policy tree | Decision rule + expected value |
| 7. **Validate** | **Run an experiment** (holdout / A/B) to confirm the causal claim | Outcome analysis / ongoing monitoring |

### Common pitfalls checklist
- [ ] Adjusting for a **mediator** or **collider** (post-treatment variables, selection on approvals/investigations)
- [ ] Poor **overlap** → extreme weights, extrapolated CATEs
- [ ] Treating a **response model** as an uplift model
- [ ] Reporting a CATE **without confidence intervals**
- [ ] Trusting a discovered graph **without expert review**
- [ ] Interference: one customer's treatment affects another (households, referral programs) — breaks SUTVA
- [ ] Time-varying treatments & feedback loops (e.g., limits adjusted repeatedly) — need panel/g-methods

---
## 5.3 🏁 Capstone exercise (take-home, ~45 min) — Proactive fee waivers & attrition

Branch staff can grant a **fee waiver** to retain customers. Leadership asks: *"Do fee waivers reduce 12-month attrition? Should we make them a systematic program — and for whom?"*

Apply the full workflow. Each step has a hidden solution.

In [ ]:
import sys, pathlib, warnings, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings("ignore"); logging.getLogger("dowhy").setLevel(logging.ERROR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from causal_utils import make_fee_waiver_data, FEE_WAIVER_EDGES, edges_to_dot, plot_dag
from lightgbm import LGBMRegressor, LGBMClassifier

fw = make_fee_waiver_data()
TRUE_ATE = fw.true_cate.mean()
data = fw.drop(columns=["true_cate"])      # what you'd have in practice
data.head()

### Step 1 — The naive analysis
Compute the difference in churn rates between customers who got a waiver and those who didn't. What would you tell leadership based on this alone?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>👀 Click to reveal a solution</b> (try it yourself first!)</summary>

```python
naive = data.loc[data.fee_waiver == 1, "churn_12m"].mean() - data.loc[data.fee_waiver == 0, "churn_12m"].mean()
print(f"Naive: waiver customers churn {naive:+.2%} MORE  →  'waivers don't work!'  (wrong)")
print(data.groupby("recent_complaint")[["fee_waiver", "churn_12m"]].mean())
# Staff target complainers, who are far more likely to churn -> confounding by indication.
```
</details>

### Step 2 — Model & identify with DoWhy
Use `FEE_WAIVER_EDGES` (plot it with `plot_dag`). Which adjustment set does DoWhy choose? Is `digital_user` in it — and should it be?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>👀 Click to reveal a solution</b> (try it yourself first!)</summary>

```python
from dowhy import CausalModel
plot_dag(FEE_WAIVER_EDGES, title="Fee waiver DAG"); plt.show()
cm = CausalModel(data=data, treatment="fee_waiver", outcome="churn_12m",
                 graph=edges_to_dot(FEE_WAIVER_EDGES))
ident = cm.identify_effect(proceed_when_unidentifiable=True)
print("Backdoor set:", ident.get_backdoor_variables())
# digital_user affects churn only (not treatment) -> not a confounder. It's not required,
# but adding outcome-only predictors is harmless and can reduce variance.
```
</details>

### Step 3 — Estimate the ATE two ways & refute
Use `backdoor.propensity_score_weighting` and `backdoor.linear_regression`. Then run a placebo refuter and a sensitivity analysis (`add_unobserved_common_cause`). Compare to `TRUE_ATE`.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>👀 Click to reveal a solution</b> (try it yourself first!)</summary>

```python
ipw = cm.estimate_effect(ident, method_name="backdoor.propensity_score_weighting")
lr = cm.estimate_effect(ident, method_name="backdoor.linear_regression")
print(f"IPW {ipw.value:+.2%} | LR {lr.value:+.2%} | TRUE ATE {TRUE_ATE:+.2%}")
print(cm.refute_estimate(ident, lr, method_name="placebo_treatment_refuter",
                         placebo_type="permute", num_simulations=5, random_seed=0))
# Note: IPW is close to the truth; the linear-probability regression is biased here because the
# effect is strongly heterogeneous (complainers vs others) and the outcome is binary.
# (We run the placebo on the regression estimate: when we tested DoWhy 0.14, its placebo
#  refuter returned a clearly non-zero placebo effect for the IPW estimator, so treat that
#  combination with caution.)
print(cm.refute_estimate(ident, lr, method_name="add_unobserved_common_cause",
                         confounders_effect_on_treatment="binary_flip",
                         confounders_effect_on_outcome="linear",
                         effect_strength_on_treatment=0.05, effect_strength_on_outcome=0.02))
```
</details>

### Step 4 — Who benefits? Estimate CATE with EconML
Fit a `CausalForestDML` with `X = [recent_complaint, balance_k, num_products, tenure, digital_user]`. Summarize the CATE by `recent_complaint` and compare with `fw.true_cate`.

In [ ]:
# YOUR CODE HERE
X_COLS = ["recent_complaint", "balance_k", "num_products", "tenure", "digital_user"]

<details>
<summary><b>👀 Click to reveal a solution</b> (try it yourself first!)</summary>

```python
from econml.dml import CausalForestDML
X_COLS = ["recent_complaint", "balance_k", "num_products", "tenure", "digital_user"]
# Tip: model RETENTION (1 - churn) so a positive CATE = benefit of the waiver
cf = CausalForestDML(model_y=LGBMRegressor(n_estimators=200, num_leaves=15, verbose=-1),
                     model_t=LGBMClassifier(n_estimators=200, num_leaves=15, verbose=-1),
                     discrete_treatment=True, cv=3, n_estimators=500, min_samples_leaf=30,
                     random_state=0)
cf.fit(1 - data.churn_12m, data.fee_waiver, X=data[X_COLS])
fw["retention_lift_hat"] = cf.effect(data[X_COLS])
fw["retention_lift_true"] = -fw.true_cate
print(fw.groupby("recent_complaint")[["retention_lift_hat", "retention_lift_true"]].mean())
print(cf.ate_inference(X=data[X_COLS]))
```
</details>

### Step 5 — Recommend a program
A waiver costs **\$60/year**; a retained customer is worth **\$900** over the horizon (illustrative). A waiver pays off when the retention lift × \$900 > \$60, i.e. when it raises retention probability by more than **6.7 pp**.
1. What share of customers qualifies? What is their complaint profile?
2. Fit a `SingleTreePolicyInterpreter` (depth 2) to produce a rule branch staff can apply.
3. Draft a 3-bullet recommendation for leadership — including how you'd **validate** it with an experiment.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>👀 Click to reveal a solution</b> (try it yourself first!)</summary>

```python
from econml.cate_interpreter import SingleTreePolicyInterpreter
VALUE, COST = 900, 60
qualifies = fw["retention_lift_hat"] * VALUE > COST
print(f"Qualifies: {qualifies.mean():.1%} of customers; complaint rate among them "
      f"{fw.loc[qualifies,'recent_complaint'].mean():.0%} vs {fw.loc[~qualifies,'recent_complaint'].mean():.0%} otherwise")
net = lambda m: ((fw.retention_lift_true * VALUE - COST) * m).sum() / 1e3
print(f"True net value — targeted program: ${net(qualifies):,.0f}K | waiver for all: ${net(np.ones(len(fw))):,.0f}K")

pt = SingleTreePolicyInterpreter(risk_level=None, max_depth=2, min_samples_leaf=200)
pt.interpret(cf, data[X_COLS], sample_treatment_costs=COST / VALUE)   # cost in probability units
plt.figure(figsize=(12, 4.5)); pt.plot(feature_names=X_COLS, treatment_names=["No waiver", "Waiver"]); plt.show()
# Recommendation sketch:
# • Waivers DO reduce attrition (~5pp on average) — the naive analysis was confounded by complaints.
# • The benefit concentrates in recently-complaining customers; a blanket program would lose money.
# • Validate with a randomized pilot: among eligible complainers, randomize waiver vs no waiver for 1 quarter.
```
</details>

---
## 5.4 Where to go next

**Books & courses (free online versions available)**
- Hernán & Robins — *Causal Inference: What If*
- Matheus Facure — *Causal Inference for the Brave and True* (Python, very practical)
- Chernozhukov, Hansen, Kallus, Spindler, Syrgkanis — *Applied Causal Inference Powered by ML and AI* (DML theory + code)
- Pearl & Mackenzie — *The Book of Why* (intuition, for stakeholders)

**Library documentation** — the PyWhy ecosystem (DoWhy, EconML, causal-learn) at `pywhy.org`; CausalML at `causalml.readthedocs.io`.

**Advanced topics we didn't cover:** difference-in-differences & synthetic control (policy changes, rollouts), regression discontinuity (score cut-offs — very relevant for credit decisions!), panel data / time-varying treatments, mediation analysis, causal representation learning, and LLM-assisted DAG construction.

---
## 🔑 The five things to remember
1. **Prediction ≠ intervention.** Ask "what happens *if we do* X?"
2. **Assumptions live in the graph** — make them explicit and review them with domain experts.
3. **Estimate with orthogonal / doubly-robust ML**, and always **refute & run sensitivity analysis**.
4. **Effects are heterogeneous** → decisions should use CATE vs cost, explained with simple policy rules.
5. **Experiments are the gold standard** — use observational causal inference to *prioritize* and *design* them, and keep control groups to *evaluate* targeting.

### 🙏 Thank you — Q&A